# Lab 7. 

In [2]:
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import seaborn as sns

Import the data in 'U.csv'

In [4]:
df = pd.read_csv('U.csv', header=None)
R = df.values.astype(float)
R

array([[5., 3., 4., ..., 0., 0., 0.],
       [4., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [5., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 5., 0., ..., 0., 0., 0.]], shape=(943, 1682))

Split your dataset:

In [5]:
def count_diffZero(row):
    cont = 0
    for i in row:
        if i != 0:
            cont += 1
    return cont

Original_row = [0, 3, 0, 5, 2, 0, 0, 0]

Let's say val = position 1 (3), test = position 4 (2), train = position 3 (5)

Train_row = [0, 0, 0, 5, 0, 0, 0, 0]

Val_row = [0, 3, 0, 0, 0, 0, 0, 0]

Test_row = [0, 0, 0, 0, 2, 0, 0, 0]

Reconstructed_row = train_row + val_row + test_row

[0, 3, 0, 5, 2, 0, 0, 0] == original_row

In [9]:
matrix_train = []
matrix_valid = []
matrix_test = []

percent_val = 0.05
percent_test = 0.05
percent_train = 1 - percent_val - percent_test

n_users, n_items = R.shape

for i in range(n_users):
    row = R[i]
    
    rated_indices = np.where(row != 0)[0]

    if len(rated_indices) < 3:
        matrix_train.append(np.zeros(n_items))
        matrix_valid.append(np.zeros(n_items))
        matrix_test.append(np.zeros(n_items))
        continue

    train_val_idx, test_idx = train_test_split(rated_indices, test_size=percent_test, random_state=42)
    train_idx, valid_idx = train_test_split(train_val_idx, test_size=percent_val / (percent_train + percent_val), random_state=42)

    train_row = np.zeros(n_items)
    valid_row = np.zeros(n_items)
    test_row = np.zeros(n_items)

    train_row[train_idx] = row[train_idx]
    valid_row[valid_idx] = row[valid_idx]
    test_row[test_idx] = row[test_idx]

    matrix_train.append(train_row)
    matrix_valid.append(valid_row)
    matrix_test.append(test_row)

R_train = np.array(matrix_train)
R_valid = np.array(matrix_valid)
R_test = np.array(matrix_test)

print("Train matrix shape:", R_train.shape)
print("Validation matrix shape:", R_valid.shape)
print("Test matrix shape:", R_test.shape)

print("Original first row:", R[0])
print("Train first row:   ", R_train[0])
print("Valid first row:   ", R_valid[0])
print("Test first row:    ", R_test[0])

reconstructed_row = R_train[0] + R_valid[0] + R_test[0]
print("Reconstructed row: ", reconstructed_row)
print("Equal to original?:", np.allclose(reconstructed_row, R[0]))

Train matrix shape: (943, 1682)
Validation matrix shape: (943, 1682)
Test matrix shape: (943, 1682)
Original first row: [5. 3. 4. ... 0. 0. 0.]
Train first row:    [5. 3. 4. ... 0. 0. 0.]
Valid first row:    [0. 0. 0. ... 0. 0. 0.]
Test first row:     [0. 0. 0. ... 0. 0. 0.]
Reconstructed row:  [5. 3. 4. ... 0. 0. 0.]
Equal to original?: True


### Initialize factors

In [ ]:
def initialize_factors(n_users, n_items, k):
    mu = 0  
    o = np.zeros(n_users)  # Bias user
    p = np.zeros(n_items)  # Bias item
    U = np.random.normal(0, 0.1, (n_users, k))
    V = np.random.normal(0, 0.1, (n_items, k))
    return mu, o, p, U, V

### Stochastic Gradient Descent - Regularization

In [21]:
def train_matrix_factorization(R_train, k=10, epochs=50, lr=0.005, lambda_reg=0.05):
    n_users, n_items = R_train.shape
    mu, o, p, U, V = initialize_factors(n_users, n_items, k)

    mu = np.mean(R_train[R_train != 0])
    observed_positions = np.array(np.where(R_train != 0)).T

    for epoch in range(epochs):
        np.random.shuffle(observed_positions)  # To make Stochastic Gradient Descent
        total_loss = 0

        for i, j in observed_positions:
            prediction = mu + o[i] + p[j] + np.dot(U[i], V[j])
            error = R_train[i, j] - prediction

            total_loss += error**2 + lambda_reg * (np.linalg.norm(U[i])**2 + np.linalg.norm(V[j])**2 + o[i]**2 + p[j]**2)
            o[i] += lr * (error - lambda_reg * o[i])
            p[j] += lr * (error - lambda_reg * p[j])

            U[i] += lr * (error * V[j] - lambda_reg * U[i])
            V[j] += lr * (error * U[i] - lambda_reg * V[j])
        rmse_epoch = np.sqrt(total_loss / len(observed_positions))
        print(f"Epoch {epoch+1}/{epochs} - RMSE (train set): {rmse_epoch:.4f}")

    return mu, o, p, U, V


### Prediction

In [22]:
def predict_all(mu, o, p, U, V):
    return mu + o[:, np.newaxis] + p[np.newaxis, :] + np.dot(U, V.T)

### Calculate RSME

In [23]:
def compute_rmse(R_true, R_pred):
    mask = R_true != 0
    mse = np.mean((R_true[mask] - R_pred[mask]) ** 2)
    return np.sqrt(mse)

### Calculate Hit Rate y ARHR in Top-K

In [24]:
def hit_rate_and_arhr(R_true, R_pred, K=10):
    n_users = R_true.shape[0]
    hit_count = 0
    reciprocal_rank_sum = 0
    n_evaluated_users = 0

    for user in range(n_users):
        true_items = np.where(R_true[user] != 0)[0]
        if len(true_items) == 0:
            continue

        predicted_scores = R_pred[user]
        recommended_items = np.argsort(predicted_scores)[::-1][:K]

        hits = np.intersect1d(recommended_items, true_items)
        if len(hits) > 0:
            hit_count += 1
            for hit in hits:
                rank = np.where(recommended_items == hit)[0][0] + 1
                reciprocal_rank_sum += 1 / rank

        n_evaluated_users += 1

    hr = hit_count / n_evaluated_users if n_evaluated_users > 0 else 0
    arhr = reciprocal_rank_sum / n_evaluated_users if n_evaluated_users > 0 else 0
    return hr, arhr


### Execution

In [25]:
mu, o, p, U, V = train_matrix_factorization(R_train, k=10, epochs=50, lr=0.005, lambda_reg=0.05)

R_pred = predict_all(mu, o, p, U, V)

print("Validation RMSE:", compute_rmse(R_valid, R_pred))
print("Test RMSE:", compute_rmse(R_test, R_pred))

hr, arhr = hit_rate_and_arhr(R_test, R_pred, K=10)
print(f"Hit Rate@10: {hr:.4f}, ARHR@10: {arhr:.4f}")


Epoch 1/50 - RMSE (train set): 1.0488
Epoch 2/50 - RMSE (train set): 0.9876
Epoch 3/50 - RMSE (train set): 0.9678
Epoch 4/50 - RMSE (train set): 0.9573
Epoch 5/50 - RMSE (train set): 0.9506
Epoch 6/50 - RMSE (train set): 0.9458
Epoch 7/50 - RMSE (train set): 0.9423
Epoch 8/50 - RMSE (train set): 0.9395
Epoch 9/50 - RMSE (train set): 0.9372
Epoch 10/50 - RMSE (train set): 0.9353
Epoch 11/50 - RMSE (train set): 0.9336
Epoch 12/50 - RMSE (train set): 0.9321
Epoch 13/50 - RMSE (train set): 0.9307
Epoch 14/50 - RMSE (train set): 0.9293
Epoch 15/50 - RMSE (train set): 0.9280
Epoch 16/50 - RMSE (train set): 0.9265
Epoch 17/50 - RMSE (train set): 0.9250
Epoch 18/50 - RMSE (train set): 0.9234
Epoch 19/50 - RMSE (train set): 0.9216
Epoch 20/50 - RMSE (train set): 0.9198
Epoch 21/50 - RMSE (train set): 0.9177
Epoch 22/50 - RMSE (train set): 0.9156
Epoch 23/50 - RMSE (train set): 0.9133
Epoch 24/50 - RMSE (train set): 0.9109
Epoch 25/50 - RMSE (train set): 0.9085
Epoch 26/50 - RMSE (train set): 0.